# agent2society — Full Feature Demo

**agent2society** is a transparent coordination layer for multi-agent systems.
Every routing decision is deterministic, scored, explained, and auditable.
No black-box LLM negotiation. No silent failures.

This notebook demonstrates every major feature in execution order:

| # | Feature | What you see |
|---|---|---|
| 1 | Build a Society & register agents | Capability graph |
| 2 | Route & dispatch a task | Score breakdown |
| 3 | Routing explanation | Human-readable rationale |
| 4 | Handoff with context chain | Intent + prior steps |
| 5 | Session trace (observability) | Full timeline |
| 6 | Governance hooks | Low-margin + conflict alerts |
| 7 | Metrics & Prometheus output | Counters + histograms |
| 8 | Persistent explanation store | Survives restarts |
| 9 | Routing optimizer | Auto-improve from labels |
| 10 | SelfAssessment (agent caveats) | Trust surface |
| 11 | Conformance guardrails | Task boundary enforcement |
| 12 | Custom framework agent (callable) | Any Python function |


In [ ]:
# Install if needed
# !pip install agent2society

# For this demo we run from source
import sys, pathlib
repo = pathlib.Path("../src")
if repo.exists() and str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import agent2society
print("agent2society", agent2society.__version__)

---
## 1. Build a Society and Register Agents

Agents are described by **Agent Cards** — lightweight JSON-ish dicts that
declare name, skills, and optionally a self-assessment (known limits).
Routing uses the text in `description` + `skills[*].description` +
optional `tags` — no LLM needed.

In [ ]:
from agent2society import Society, Handoff, SessionTracer

# ---- define agent cards ---------------------------------------------------
def card(name, description, skill_desc, tags, *, caveats=None):
    c = {
        "name": name,
        "description": description,
        "url": f"local://{name}",
        "version": "1.0.0",
        "skills": [{
            "id": name.lower().replace(" ", "_"),
            "name": name,
            "description": skill_desc,
            "tags": tags,
        }],
        "defaultInputModes": ["text"],
        "defaultOutputModes": ["text"],
    }
    if caveats:
        c["selfAssessment"] = {"knownLimitations": caveats}
    return c

AGENTS = [
    card("DataAnalyst",
         "Statistical analysis, data science, machine learning insights",
         "Analyse datasets, compute statistics, detect trends and anomalies",
         ["data", "statistics", "analysis", "ml", "pandas", "sql"],
         caveats=["Does not write production code", "Requires clean tabular data"]),

    card("ReportWriter",
         "Executive report writing, business memos, narrative summaries",
         "Draft executive reports, write business memos, summarise findings",
         ["writing", "report", "memo", "executive", "summary"]),

    card("CodeReviewer",
         "Code review, security audit, Python and TypeScript quality checks",
         "Review code for bugs, security issues, and style violations",
         ["code", "review", "python", "typescript", "security", "audit"],
         caveats=["Cannot run code", "Best on files under 500 lines"]),

    card("CustomerSupport",
         "Handle customer enquiries, refund requests, product questions",
         "Respond to customer support tickets and resolve common issues",
         ["customer", "support", "refund", "ticket", "enquiry"]),

    card("SQLAgent",
         "Write and optimise SQL queries for database interrogation",
         "Generate SQL queries, explain query plans, optimise slow queries",
         ["sql", "database", "query", "postgres", "mysql", "optimise"]),
]

# ---- build the Society ----------------------------------------------------
s = Society(strict=False, min_score=0.05)
for a in AGENTS:
    s.add(a)

print("Agents registered:", s.agents())

In [ ]:
# Register local handlers so dispatch works in this notebook
# (in production these are real A2A endpoints or wrapped framework agents)
RESPONSES = {
    "local://DataAnalyst":    "Churn rate is 8.3% (↑2.1pp MoM). Top driver: plan downgrade after 60-day trial.",
    "local://ReportWriter":   "Executive Summary: Q3 churn increased to 8.3%. Recommend targeted retention campaign.",
    "local://CodeReviewer":   "Line 42: SQL injection risk. Line 87: unused import. Overall: 3 high, 2 medium issues.",
    "local://CustomerSupport":"Hi! I've processed your refund request. Amount will appear in 3-5 business days.",
    "local://SQLAgent":       "SELECT customer_id, COUNT(*) as orders FROM orders GROUP BY customer_id HAVING COUNT(*) > 10;",
}

for url, response in RESPONSES.items():
    s._local.register(url, lambda u, p, r=response: {
        "jsonrpc": "2.0", "id": p.get("id", "0"),
        "result": {"status": "completed",
                   "artifacts": [{"parts": [{"type": "text", "text": r}]}]}
    })

print("Local handlers registered.")

---
## 2. Route and Dispatch a Task

`.route()` shows the scoring without dispatching. `.run()` routes + dispatches.

In [ ]:
task = "Analyse customer churn data and identify the main drivers"

# Preview routing without dispatching
candidates = s.route(task, top_k=5)
print(f"Top candidates for: {task!r}\n")
for c in candidates:
    print(f"  {c.agent:20s}  score={c.score:.4f}  "
          f"semantic={c.semantic:.4f}  tags={c.tag_overlap:.4f}  "
          f"matched={c.matched_tokens[:3]}")

In [ ]:
# Actually dispatch
result = s.run(task)
print("Response:", result)

---
## 3. Routing Explanation — the WHY behind every decision

Every `.run()` call produces a `RoutingExplanation` stored by handoff id.
It shows: chosen agent, score, margin, alternatives considered, tokens
that drove the match, and the agent's self-declared caveats.

In [ ]:
exp = s.last_explanation()

# Human-readable render
print(exp.render())

In [ ]:
# Structured dict — JSON-serialisable, easy to store or ship to a dashboard
import json
print(json.dumps(exp.to_dict(), indent=2))

In [ ]:
# Key fields at a glance
print(f"Chosen agent : {exp.chosen_agent} :: {exp.chosen_skill}")
print(f"Confidence   : {exp.confidence:.4f}")
print(f"Margin       : {exp.margin:.4f}  (gap to runner-up)")
print(f"Flags        : {exp.flags or '(none)'}")
print(f"Rationale    : {exp.rationale}")
print(f"Agent caveats: {exp.agent_self_caveats}")

---
## 4. Handoff — carry intent and prior context across agents

A `Handoff` wraps a task with:
- `intent` — machine-readable verb ("analyse", "summarise", …)
- `assumptions` — constraints the caller has already checked
- `prior` — chain of `DecisionRecord`s from previous agents
- `confidence_required` — minimum score to proceed

In [ ]:
from agent2society import Handoff, DecisionRecord

# Step 1: analyse
h1 = Handoff(
    task="Compute churn statistics for Q3 cohort",
    intent="analyse",
    assumptions=["Data is clean", "Q3 = Jul-Sep 2025"],
)
r1 = s.run(h1)
print("Step 1 result:", r1)

exp1 = s.explain(h1.id)
print(f"  -> routed to {exp1.chosen_agent}  (score={exp1.confidence:.3f})")

In [ ]:
# Step 2: write executive report — chain the prior decision
prior_record = DecisionRecord(
    agent=exp1.chosen_agent,
    skill=exp1.chosen_skill,
    summary=r1[:80],
)

h2 = Handoff(
    task="Write an executive memo summarising the churn analysis",
    intent="summarise",
    prior=[prior_record],
    confidence_required=0.3,   # raise the bar for this step
)
r2 = s.run(h2)
print("Step 2 result:", r2)

exp2 = s.explain(h2.id)
print(f"\n{exp2.render()}")

---
## 5. Session Trace — full observability timeline

`SessionTracer` joins every `run()` call into one ordered timeline.
It shows what happened, why, and whether any quality flags fired.

In [ ]:
# Run a few more tasks to make the trace interesting
s.run("Review this Python file for security vulnerabilities")
s.run("Write a SQL query to find top 10 customers by order count")
s.run("Process refund for order #A-12345")   # customer support
s.run("xyzzy frobnicate plugh")              # intentionally unroutable (OOD)

tracer = SessionTracer(s)
tracer.print()

In [ ]:
# Summary aggregates
import json
print(json.dumps(tracer.summary(), indent=2))

In [ ]:
# Iterate events programmatically
for evt in tracer.events():
    flag_str = f"  [{', '.join(evt.flags)}]" if evt.flags else ""
    agent = evt.chosen_agent or "UNROUTED"
    score = f"{evt.score:.3f}" if evt.score else "  n/a"
    print(f"[{evt.seq}] {agent:20s}  score={score}  margin={evt.margin:.3f}{flag_str}")
    print(f"    {evt.task[:70]}")

---
## 6. Governance Hooks — real-time routing quality alerts

Hooks fire as **side effects** — they never block or alter dispatch.
Detection-only by design: the package surfaces the signal; your code decides.

Available hooks:
- `on_low_margin` — score gap between top-1 and top-2 is narrow
- `on_low_confidence` — chosen score below a threshold
- `on_conflict` — same task text routed to different agents across calls
- `on_capability_drift` — one agent chosen for too many distinct skill types
- `on_human_review` — caller flags result for human review

In [ ]:
from agent2society import Society, Handoff

alerts = []   # collect all hook events here

s2 = Society(strict=False, min_score=0.05)
for a in AGENTS:
    s2.add(a)
for url, response in RESPONSES.items():
    s2._local.register(url, lambda u, p, r=response: {
        "jsonrpc": "2.0", "id": p.get("id", "0"),
        "result": {"status": "completed",
                   "artifacts": [{"parts": [{"type": "text", "text": r}]}]}
    })

# Hook: low margin (score gap < 0.15 — wide threshold to see it fire)
s2.on_low_margin(
    lambda exp: alerts.append(("LOW_MARGIN", exp.chosen_agent, f"margin={exp.margin:.3f}")),
    threshold=0.15,
)

# Hook: low confidence (chosen score < 0.4)
s2.on_low_confidence(
    lambda exp: alerts.append(("LOW_CONFIDENCE", exp.chosen_agent, f"score={exp.confidence:.3f}")),
    threshold=0.4,
)

# Hook: conflict detector (same task -> different agents in window)
s2.on_conflict(
    lambda c: alerts.append(("CONFLICT", c.kind, c.detail))
)

# Hook: capability drift (one agent chosen for 3+ distinct skills)
s2.on_capability_drift(
    lambda d: alerts.append(("DRIFT", d.agent, d.detail))
)

# Run several tasks
tasks = [
    "Compute monthly revenue statistics",
    "Write an executive summary of sales performance",
    "Review the authentication module for SQL injection",
    "Handle a refund complaint from customer #4421",
    "Write the best SQL query to find duplicate orders",
]
for t in tasks:
    s2.run(t)

print(f"Alerts fired: {len(alerts)}")
for kind, subject, detail in alerts:
    print(f"  [{kind}] {subject}  —  {detail}")

---
## 7. Metrics — Prometheus-compatible counters and histograms

Every `Society` carries a `MetricsCollector`. No external dependency — the
output is standard Prometheus text exposition, readable by any scrape stack.

In [ ]:
snap = s2.metrics.snapshot()
print("--- Counters ---")
for name, series in sorted(snap["counters"].items()):
    total = sum(e["value"] for e in series)
    if total > 0:
        print(f"  {name}: {total}")

In [ ]:
print("--- Histograms ---")
for name, series in sorted(snap["histograms"].items()):
    for e in series:
        if e["count"] > 0:
            print(f"  {name}: count={e['count']}  avg={e['avg']:.4f}  "
                  f"min={e['min']:.4f}  max={e['max']:.4f}")

In [ ]:
# Prometheus text exposition — drop behind any scraper
prom = s2.metrics.render_prometheus()
# Show first 40 lines
for line in prom.splitlines()[:40]:
    print(line)

---
## 8. Persistent Explanation Store — survives process restarts

`JsonlFileStore` appends each explanation as one JSON line. On restart it
replays the file and re-builds the index. Corrupt lines are skipped.

In [ ]:
import tempfile, pathlib
from agent2society import Society, Handoff, JsonlFileStore

store_path = pathlib.Path(tempfile.mktemp(suffix=".jsonl"))

# Session A — write explanations to disk
sA = Society(strict=False, store=JsonlFileStore(store_path))
for a in AGENTS:
    sA.add(a)
for url, response in RESPONSES.items():
    sA._local.register(url, lambda u, p, r=response: {
        "jsonrpc": "2.0", "id": p.get("id", "0"),
        "result": {"status": "completed",
                   "artifacts": [{"parts": [{"type": "text", "text": r}]}]}
    })

h = Handoff(task="Analyse revenue trends for Q3")
sA.run(h)

print(f"Store path  : {store_path}")
print(f"File size   : {store_path.stat().st_size} bytes")
print(f"Explanations: {len(sA.explanations())}")
print(f"Handoff id  : {h.id}")

In [ ]:
# Session B — new Society, same file; loads past explanations
sB = Society(strict=False, store=JsonlFileStore(store_path))
recovered = sB.explain(h.id)

print("Recovered explanation:")
print(recovered.render())

store_path.unlink()   # cleanup

---
## 9. Routing Optimizer — improve routing from labelled observations

`Society.optimize(labels)` mines discriminative tokens from past routing
decisions and proposes new skill tags. Every proposal is backtested;
only changes that fix more routes than they break are accepted.
Nothing changes until you call `apply_optimization(report)`.

In [ ]:
from agent2society import Society, Handoff

# Simulate a small labelled dataset:
# (handoff_id, correct_agent, correct_skill)
s3 = Society(strict=False, min_score=0.0)
for a in AGENTS:
    s3.add(a)
for url, response in RESPONSES.items():
    s3._local.register(url, lambda u, p, r=response: {
        "jsonrpc": "2.0", "id": p.get("id", "0"),
        "result": {"status": "completed",
                   "artifacts": [{"parts": [{"type": "text", "text": r}]}]}
    })

# Run some tasks and collect handoff ids
handoffs = []
tasks = [
    ("Compute statistical correlations in the sales dataset", "DataAnalyst", "dataanalyst"),
    ("Draft a memo on the Q3 performance findings",           "ReportWriter", "reportwriter"),
    ("Run a SQL aggregation on orders",                       "SQLAgent",     "sqlagent"),
]
for task, correct_agent, correct_skill in tasks:
    h = Handoff(task=task)
    s3.run(h)
    handoffs.append((h.id, correct_agent, correct_skill))

# Diagnose before optimizing
print("Before optimization:")
for hid, ca, cs in handoffs:
    exp = s3.explain(hid)
    match = "OK" if exp.chosen_agent == ca else "MISS"
    print(f"  [{match}] {exp.task[:50]:50s}  -> {exp.chosen_agent}  (expected {ca})")

In [ ]:
# Run the optimizer
report = s3.optimize(handoffs)

print(f"Proposed edits : {len(report.edits)}")
print(f"Accepted edits : {len(report.accepted_edits)}")
for edit in report.accepted_edits:
    print(f"  {edit.agent} :: {edit.skill_id}  +tags={edit.add_tags}")
    print(f"    fixes={edit.fixes}  regressions={edit.regressions}")

In [ ]:
# Apply and re-check
applied = s3.apply_optimization(report)
print(f"Applied {applied} edits\n")

print("After optimization:")
for task, correct_agent, correct_skill in tasks:
    h = Handoff(task=task)
    s3.run(h)
    exp = s3.explain(h.id)
    match = "OK" if exp.chosen_agent == correct_agent else "MISS"
    print(f"  [{match}] {exp.task[:50]:50s}  -> {exp.chosen_agent}")

---
## 10. SelfAssessment — agent-declared trust surface

Each agent can declare its own limits in the card. These appear in the
explanation under `agent_self_caveats` — visible to every caller without
requiring a separate API call.

In [ ]:
from agent2society import Society, Handoff, SelfAssessment

cautious_card = {
    "name": "MedicalSummariser",
    "description": "Summarise clinical notes and lab reports",
    "url": "local://MedicalSummariser",
    "version": "1.0.0",
    "skills": [{
        "id": "summarise_clinical",
        "name": "Summarise Clinical Notes",
        "description": "Summarise patient notes, lab results, radiology reports",
        "tags": ["medical", "clinical", "summarise", "lab", "radiology"],
    }],
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text"],
    "selfAssessment": {
        "knownLimitations": [
            "Not a substitute for clinical judgement",
            "Cannot interpret images or PDFs",
        ],
        "escalateWhen": [
            "Patient mentions self-harm",
            "Abnormal lab values outside reference range",
        ],
        "outOfScope": [
            "Prescribing medication",
            "Diagnosing conditions",
        ],
    },
}

sm = Society(strict=False)
sm.add(cautious_card)
sm._local.register("local://MedicalSummariser", lambda u, p: {
    "jsonrpc": "2.0", "id": p.get("id", "0"),
    "result": {"status": "completed",
               "artifacts": [{"parts": [{"type": "text", "text": "Summary: BP 120/80, all labs WNL."}]}]}
})

h = Handoff(task="Summarise the latest lab results for patient P-99")
sm.run(h)
exp = sm.explain(h.id)
print(exp.render())

---
## 11. Conformance Guardrails — enforce task boundaries

Each agent node can have an `allow` list (task must mention at least one
term) or a `deny` list (task must not mention any term). Violations are
recorded in the explanation and block dispatch (when `strict=True`).

In [ ]:
from agent2society import Society, Handoff

sg = Society(strict=False, min_score=0.0)
sg.add(card("LegalAgent", "Legal research and contract review",
            "Review contracts and summarise legal risks",
            ["legal", "contract", "compliance", "risk"]))

sg._local.register("local://LegalAgent", lambda u, p: {
    "jsonrpc": "2.0", "id": p.get("id", "0"),
    "result": {"status": "completed",
               "artifacts": [{"parts": [{"type": "text", "text": "Contract review complete."}]}]}
})

# Deny: this agent must never touch PII or HR data
sg.boundary("LegalAgent", deny=["salary", "personal data", "GDPR breach"])

# Good task — allowed
h_ok = Handoff(task="Review the SaaS vendor contract for indemnity clauses")
sg.run(h_ok)
exp_ok = sg.explain(h_ok.id)
print("Allowed task:")
print(exp_ok.render())
print()

In [ ]:
# Blocked task — hits the deny list
h_bad = Handoff(task="Review the contract for personal data handling and GDPR breach clauses")
sg.run(h_bad)   # strict=False so no exception — returns ""
exp_bad = sg.explain(h_bad.id)
print("Blocked task:")
print(exp_bad.render())

---
## 12. Custom Framework Agent — wrap any callable

Passing any Python callable (or a LangGraph / CrewAI / AutoGen object)
to `Society.add()` auto-adapts it. The adapter extracts skills from the
object's structure and registers a local handler automatically.

Here we demonstrate with a plain Python function — the simplest case.

In [ ]:
from agent2society import Society, Handoff

# A plain Python function — the callable adapter wraps it automatically
def my_translation_agent(task: str) -> str:
    """translate text into Spanish"""
    return f"[ES] (simulated translation of: {task[:40]})"

sc = Society(strict=False)
card_obj = sc.add(my_translation_agent)
print("Auto-generated card:")
print(f"  name  : {card_obj.name}")
print(f"  url   : {card_obj.url}")
print(f"  skills: {[s.id for s in card_obj.skills]}")

In [ ]:
h = Handoff(task="Translate this customer feedback message to Spanish")
result = sc.run(h)
print("Result:", result)
print()
print(sc.explain(h.id).render())

---
## Summary — what you just saw

| Capability | How to use |
|---|---|
| Register agents | `s.add(card_dict)` or `s.add(any_callable)` |
| Route without dispatch | `s.route(task, top_k=5)` |
| Route + dispatch | `s.run(task)` or `s.run(Handoff(...))` |
| Per-decision explanation | `s.explain(handoff_id)` or `s.last_explanation()` |
| Session trace | `SessionTracer(s).print()` |
| Governance hooks | `s.on_low_margin(fn, threshold=0.1)` etc. |
| Metrics | `s.metrics.snapshot()` / `render_prometheus()` |
| Persistent store | `Society(store=JsonlFileStore(path))` |
| Routing optimizer | `report = s.optimize(labels); s.apply_optimization(report)` |
| Agent trust surface | `selfAssessment` field in card |
| Conformance guardrails | `s.boundary(agent, allow=[...], deny=[...])` |
| Framework adapters | `s.add(langgraph_graph)` / `s.add(crew)` / `s.add(fn)` |

Every routing decision is:
- **Deterministic** — same inputs always produce same routing
- **Explained** — rationale in human-readable text + structured dict
- **Auditable** — stored, queryable, JSON-serialisable
- **Observable** — metrics, trace, governance hooks, Prometheus output
- **Trustable** — agent self-caveats surface at decision time, not buried in docs
